# Notebook 06 — Module 3: semantic intent verification


In [1]:
# ── Cell 1 · Configuration ────────────────────────────────────────────────────
CONFIG = {
    "SEED": 42,


    "TAU_INTENT": 0.30,
    "TAU_SWEEP": [0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90],

    "EMBED_MODEL": "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext",
    "MAX_LEN": 512,
    "BATCH_SIZE": 32,


    "OVERRIDE_MARKERS": [
        "ignore previous", "ignore all previous", "ignore the above",
        "disregard the", "disregard all", "system override", "new instruction",
        "override:", "admin:", "[admin]", "[config]", "[system]",
        "forget everything", "forget the", "prior context is void",
        "end of document", "attention model", "do not follow",
        "your reply must", "you must instead", "instead output",
    ],

    "USE_DRIVE":      True,
    "DRIVE_DIR":      "/content/drive/MyDrive/ClinicalShield_v2",
    "PUSH_TO_GITHUB": True,
    "GITHUB_REPO":    "NehlTech/ClinicalShield",
    "GITHUB_BRANCH":  "v2-revision",
    "GIT_USER_NAME":  "Adu-Boahene Bright",
    "GIT_USER_EMAIL": "baduboahene@st.knust.edu.gh",
}
SEED = CONFIG["SEED"]
print("tau (from v1)   :", CONFIG["TAU_INTENT"])
print("markers         :", len(CONFIG["OVERRIDE_MARKERS"]))
print("operating point chosen on validation, reported on test")


tau (from v1)   : 0.3
markers         : 21
operating point chosen on validation, reported on test


In [2]:
!rm -rf /content/ClinicalShield

In [3]:
# ── Cell 2 · Environment ───────────────────────────────────────────
import sys, os, json, random, hashlib, subprocess, shutil, time, math
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np

random.seed(SEED); np.random.seed(SEED)

IN_COLAB = "google.colab" in sys.modules
DRIVE_ROOT, REPO_DIR = None, None

if IN_COLAB:
    if CONFIG["USE_DRIVE"]:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        DRIVE_ROOT = Path(CONFIG["DRIVE_DIR"]); DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
        print("drive :", DRIVE_ROOT)
    repo_name = CONFIG["GITHUB_REPO"].split("/")[-1]
    REPO_DIR = Path("/content") / repo_name
    if not REPO_DIR.exists():
        try:
            from google.colab import userdata
            tok = userdata.get("GITHUB_TOKEN")
            r = subprocess.run(["git","clone","-q",
                "https://" + tok + "@github.com/" + CONFIG["GITHUB_REPO"] + ".git",
                str(REPO_DIR)], capture_output=True, text=True)
            print("clone :", "ok" if r.returncode == 0 else r.stderr[:200])
        except Exception as e:
            print("clone skipped:", type(e).__name__)
    else:
        print("clone : already present")
    if REPO_DIR.exists():
        for k, v in [("user.name", CONFIG["GIT_USER_NAME"]),
                     ("user.email", CONFIG["GIT_USER_EMAIL"])]:
            subprocess.run(["git","-C",str(REPO_DIR),"config",k,v], check=False)
        subprocess.run(["git","-C",str(REPO_DIR),"checkout","-q",
                        CONFIG["GITHUB_BRANCH"]], check=False, capture_output=True)
        subprocess.run(["git","-C",str(REPO_DIR),"pull","-q","origin",
                        CONFIG["GITHUB_BRANCH"]], check=False, capture_output=True)
    ROOT = REPO_DIR if REPO_DIR.exists() else Path("/content")
else:
    ROOT = Path.cwd()
    while not (ROOT/".git").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT/".git").exists():
        ROOT = Path.cwd()

DIRS = {"dataset": ROOT/"data"/"dataset", "stats": ROOT/"data"/"stats",
        "results": ROOT/"data"/"results", "src": ROOT/"src"/"clinicalshield"}
for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)
print("root  :", ROOT)

for pkg, mod in [("transformers","transformers"), ("torch","torch"),
                 ("scikit-learn","sklearn")]:
    try:
        __import__(mod)
    except ImportError:
        subprocess.run([sys.executable,"-m","pip","install","-q",pkg], check=False)

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

def read_jsonl(p):
    with open(p, encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]

REQUIRED_FIELDS = {"split", "template_id"}

def _fields_of(path):
    with open(path, encoding="utf-8") as f:
        return set(json.loads(f.readline()).keys())

def restore_fresh(rel_path, label):
    candidates = []
    local = ROOT / rel_path
    if local.exists():
        candidates.append((local, "repo"))
    if DRIVE_ROOT and (DRIVE_ROOT / rel_path).exists():
        candidates.append((DRIVE_ROOT / rel_path, "drive"))
    if not candidates:
        raise FileNotFoundError("\n%s not found in repo or Drive. Run NB02/NB03." % label)
    for path, src in candidates:
        missing = REQUIRED_FIELDS - _fields_of(path)
        if not missing:
            if src == "drive":
                local.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(path, local)
            return local, src
        print("  stale copy ignored (%s): missing %s" % (src, sorted(missing)))
    raise RuntimeError("\nEvery copy of %s predates the template-disjoint fix." % label)

p_ds, src_ds = restore_fresh("data/dataset/attack_dataset_split.jsonl",
                             "attack_dataset_split.jsonl")
dataset = read_jsonl(p_ds)

print("")
print("dataset from %s  %d chunks" % (src_ds, len(dataset)))
for s in ["train","val","test"]:
    rows = [d for d in dataset if d["split"] == s]
    print("  %-6s %5d chunks  %4d adversarial"
          % (s, len(rows), sum(d["label"] for d in rows)))


Mounted at /content/drive
drive : /content/drive/MyDrive/ClinicalShield_v2
clone : ok
root  : /content/ClinicalShield
device: cpu

dataset from repo  5353 chunks
  train   3840 chunks  1720 adversarial
  val      812 chunks   360 adversarial
  test     701 chunks   328 adversarial


In [4]:
# ── Cell 3 ──────────────────────────


SECTION_QUERY = {
    "indications_and_usage":       "What is {drug} indicated for?",
    "dosage_and_administration":   "What is the recommended dosing for {drug}?",
    "contraindications":           "What are the contraindications for {drug}?",
    "warnings_and_cautions":       "What warnings apply to {drug}?",
    "drug_interactions":           "What drug interactions are known for {drug}?",
    "boxed_warning":               "Does {drug} carry a boxed warning?",
    "adverse_reactions":           "What adverse reactions are reported for {drug}?",
    "use_in_specific_populations": "Is {drug} safe in pregnancy or renal impairment?",
}

for d in dataset:
    tpl = SECTION_QUERY.get(d["section"], "What clinical guidance applies to {drug}?")
    d["query"] = tpl.format(drug=d["drug"])

parts = {s: [d for d in dataset if d["split"] == s] for s in ["train", "val", "test"]}

print("queries built from host metadata only (drug + section), never from payload")
print("")
from collections import Counter as _C
for sec, n in sorted(_C(d["section"] for d in dataset).items()):
    print("  %-30s %5d  ->  %s"
          % (sec, n, SECTION_QUERY.get(sec, "(generic)").format(drug="X")[:44]))


queries built from host metadata only (drug + section), never from payload

  adverse_reactions               1256  ->  What adverse reactions are reported for X?
  boxed_warning                    125  ->  Does X carry a boxed warning?
  contraindications                150  ->  What are the contraindications for X?
  dosage_and_administration        987  ->  What is the recommended dosing for X?
  drug_interactions                560  ->  What drug interactions are known for X?
  indications_and_usage            324  ->  What is X indicated for?
  use_in_specific_populations      830  ->  Is X safe in pregnancy or renal impairment?
  warnings_and_cautions           1121  ->  What warnings apply to X?


In [5]:
# ── Cell 4 · Embeddings and cosine similarity ─────────────────────────────────
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import roc_auc_score

tok = AutoTokenizer.from_pretrained(CONFIG["EMBED_MODEL"])
emb_model = AutoModel.from_pretrained(CONFIG["EMBED_MODEL"]).to(DEVICE).eval()
if DEVICE == "cuda":
    emb_model = emb_model.half()          # fp16 roughly doubles throughput

BATCH = 64 if DEVICE == "cuda" else 8
if DEVICE != "cuda":
    print("WARNING: no GPU. Embedding 5,353 documents on CPU takes ~1 hour.")
    print("         Switch to a GPU runtime and re-run from Cell 1.")

@torch.no_grad()
def mean_pooled(texts, batch=None, desc=""):
    """Mean-pooled token embeddings, masked so padding does not dilute the vector.

    Texts are processed in length order so each batch pads to a similar length.
    With document lengths ranging from 60 to 250 words, sorting cuts wasted
    compute substantially; the original order is restored before returning.
    """
    batch = batch or BATCH
    order = sorted(range(len(texts)), key=lambda i: len(texts[i]))
    out = [None] * len(texts)
    t0 = time.time()
    for s in range(0, len(order), batch):
        idx = order[s:s+batch]
        enc = tok([texts[i] for i in idx], truncation=True, padding=True,
                  max_length=CONFIG["MAX_LEN"], return_tensors="pt").to(DEVICE)
        h = emb_model(**enc).last_hidden_state
        m = enc["attention_mask"].unsqueeze(-1).to(h.dtype)
        v = (h * m).sum(1) / m.sum(1).clamp(min=1e-9)
        v = torch.nn.functional.normalize(v.float(), dim=-1).cpu()
        for k, i in enumerate(idx):
            out[i] = v[k]
        done = s + len(idx)
        if desc and (done % (batch*20) == 0 or done >= len(order)):
            el = time.time() - t0
            eta = el / done * (len(order) - done)
            print("  %s %d/%d  %.0fs elapsed, ~%.0fs left"
                  % (desc, done, len(order), el, eta), end="\r")
    if desc:
        print()
    return torch.stack(out)

t0 = time.time()
doc_vecs = mean_pooled([d["text"] for d in dataset], desc="documents")
qry_vecs = mean_pooled([d["query"] for d in dataset], desc="queries  ")
cos = (doc_vecs * qry_vecs).sum(-1).numpy()
for d, c in zip(dataset, cos):
    d["cosine"] = float(c)
print("embedded %d document-query pairs in %.1f min"
      % (len(dataset), (time.time()-t0)/60))

ben = np.array([d["cosine"] for d in dataset if d["label"] == 0])
adv = np.array([d["cosine"] for d in dataset if d["label"] == 1])
print("")
print("cosine similarity, query vs document")
print("  benign      mean %.4f  sd %.4f  [%.3f, %.3f]"
      % (ben.mean(), ben.std(), np.percentile(ben,5), np.percentile(ben,95)))
print("  adversarial mean %.4f  sd %.4f  [%.3f, %.3f]"
      % (adv.mean(), adv.std(), np.percentile(adv,5), np.percentile(adv,95)))
print("  separation  %.4f" % abs(ben.mean() - adv.mean()))

y = np.array([d["label"] for d in dataset])
auc_sim = roc_auc_score(y, -np.array([d["cosine"] for d in dataset]))
print("  AUC using similarity alone : %.4f  (0.50 = no signal)" % auc_sim)

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/226k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  440MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

KeyboardInterrupt: 

## Choosing the threshold on validation

Two sweeps: similarity alone, then similarity combined with the markers. Both on validation.

Running the sweep on test and then reporting the best point would be selecting the operating
point to flatter the result — a subtler version of the error Reviewer 4 caught in v1.

In [ ]:
# ── Cell 5 · Threshold selection on VALIDATION ────────────────────────────────
# v1 used tau = 0.30 with no stated justification. Here the sweep runs on the
# validation partition, the operating point is fixed, and test is evaluated once
# at that point -- so the reported number is not chosen to flatter itself.

def flag(d, tau, use_markers=True):
    """Module 3 raises a flag on either signal: the document sits too far from
    the query in meaning, or it carries a known override marker."""
    if d["cosine"] < tau:
        return True, "similarity"
    if use_markers:
        low = d["text"].lower()
        for m in CONFIG["OVERRIDE_MARKERS"]:
            if m in low:
                return True, "marker"
    return False, None

def score(rows, tau, use_markers=True):
    tp = fp = fn = tn = 0
    by_reason = Counter()
    for d in rows:
        f, why = flag(d, tau, use_markers)
        if f:
            by_reason[why] += 1
        if d["label"] == 1:
            tp += f; fn += (not f)
        else:
            fp += f; tn += (not f)
    prec = tp / (tp + fp) if (tp + fp) else 0.0
    rec = tp / (tp + fn) if (tp + fn) else 0.0
    return {"tp": tp, "fp": fp, "fn": fn, "tn": tn,
            "precision": prec, "recall": rec,
            "f1": 2*prec*rec/(prec+rec) if (prec+rec) else 0.0,
            "fpr": fp/(fp+tn) if (fp+tn) else 0.0,
            "reasons": dict(by_reason)}

print("VALIDATION sweep — similarity threshold only (markers disabled)")
print("  tau     recall    FPR      F1")
sim_only = {}
for tau in CONFIG["TAU_SWEEP"]:
    s = score(parts["val"], tau, use_markers=False)
    sim_only[tau] = s
    print("  %.2f   %.4f   %.4f   %.4f" % (tau, s["recall"], s["fpr"], s["f1"]))

print("")
print("VALIDATION — similarity + override markers")
print("  tau     recall    FPR      F1")
combined = {}
for tau in CONFIG["TAU_SWEEP"]:
    s = score(parts["val"], tau, use_markers=True)
    combined[tau] = s
    print("  %.2f   %.4f   %.4f   %.4f" % (tau, s["recall"], s["fpr"], s["f1"]))

# ── operating point selection ────────────────────────────────────────────────
# Selecting by F1 is wrong here. Benign and adversarial cosines overlap heavily,
# so F1 is maximised by driving the threshold up and flagging almost everything:
# in development that produced tau = 0.90 at a 79% false positive rate. A CDSS
# that flags four documents in five is not deployable, and the paper's own
# argument is that false positives matter more than detection rate.
#
# The threshold is therefore chosen to maximise recall subject to an explicit
# false-positive budget, stated in advance rather than discovered afterwards.

FPR_BUDGET = 0.05

feasible = {t: s for t, s in combined.items() if s["fpr"] <= FPR_BUDGET}
if feasible:
    best_tau = max(feasible, key=lambda t: feasible[t]["recall"])
    basis = "max recall subject to FPR <= %.0f%%" % (100 * FPR_BUDGET)
else:
    best_tau = min(combined, key=lambda t: combined[t]["fpr"])
    basis = "no threshold met the FPR budget; lowest-FPR point selected"

print("")
print("OPERATING POINT")
print("  criterion       : %s" % basis)
print("  FPR budget      : %.0f%%" % (100 * FPR_BUDGET))
print("  thresholds meeting budget : %d of %d" % (len(feasible), len(combined)))
print("  selected tau    : %.2f   (v1 used %.2f)" % (best_tau, CONFIG["TAU_INTENT"]))
s_sel = combined[best_tau]
print("  validation      : recall %.4f  FPR %.4f  precision %.4f"
      % (s_sel["recall"], s_sel["fpr"], s_sel["precision"]))

f1_tau = max(combined, key=lambda t: combined[t]["f1"])
print("")
print("  for contrast, the F1-optimal point would be tau %.2f at FPR %.4f"
      % (f1_tau, combined[f1_tau]["fpr"]))


## Held-out evaluation, with the signals separated

Three configurations at the validation-selected threshold: similarity only, markers only, and
both.

Reporting them separately is the point. If markers-only matches both-combined, then the
similarity check contributes nothing and the paper should say so rather than describe a
two-signal module that is effectively one.

In [ ]:
# ── Cell 6 · Held-out evaluation, and what each signal contributes ────────────
# Evaluated once, at the validation-selected operating point.

TAU = best_tau
test_combined = score(parts["test"], TAU, use_markers=True)
test_sim_only = score(parts["test"], TAU, use_markers=False)
test_marker_only = score(parts["test"], -1.0, use_markers=True)   # tau unreachable

print("HELD-OUT TEST at tau = %.2f (selected on validation)" % TAU)
print("")
print("configuration          recall     FPR       precision   F1")
for name, s in [("similarity only", test_sim_only),
                ("markers only", test_marker_only),
                ("both (Module 3)", test_combined)]:
    print("  %-20s %.4f   %.4f    %.4f     %.4f"
          % (name, s["recall"], s["fpr"], s["precision"], s["f1"]))

print("")
print("which signal fired, among detections")
print("  ", test_combined["reasons"])

# Recall by vector: the finding v1 reported on MPIB, reproduced on own data
print("")
print("recall by attack vector, Module 3 alone")
for vec in ["override", "misinformation"]:
    rows = [d for d in parts["test"] if d["label"] == 1 and d["vector"] == vec]
    if not rows:
        continue
    hit = sum(flag(d, TAU)[0] for d in rows)
    print("  %-16s n=%4d  recall %.4f" % (vec, len(rows), hit/len(rows)))

print("")
print("recall by category, Module 3 alone")
for cat in sorted({d["category"] for d in parts["test"] if d["label"] == 1}):
    rows = [d for d in parts["test"] if d["label"] == 1 and d["category"] == cat]
    hit = sum(flag(d, TAU)[0] for d in rows)
    print("  %-28s n=%4d  recall %.4f" % (cat, len(rows), hit/len(rows)))


## Export and persist

In [ ]:
# ── Cell 7 · Export Module 3 and save results ─────────────────────────────────
MODULE3_SRC = '''"""ClinicalShield Module 3 — semantic intent verification.

Flags a retrieved document when it diverges from the clinical query in meaning,
or when it carries a known instruction-override marker. Generated by
notebooks/06_module3_intent.ipynb.
"""

import torch
import torch.nn.functional as F

OVERRIDE_MARKERS = %s

TAU_INTENT = %.2f


def mean_pooled(texts, tokenizer, model, device="cpu", max_len=512, batch=32):
    out = []
    with torch.no_grad():
        for i in range(0, len(texts), batch):
            enc = tokenizer(texts[i:i+batch], truncation=True, padding=True,
                            max_length=max_len, return_tensors="pt").to(device)
            h = model(**enc).last_hidden_state
            m = enc["attention_mask"].unsqueeze(-1).float()
            v = (h * m).sum(1) / m.sum(1).clamp(min=1e-9)
            out.append(F.normalize(v, dim=-1).cpu())
    return torch.cat(out)


def module3(document, query, cosine=None, tau=TAU_INTENT):
    """Returns (flagged, reason). Cosine may be precomputed."""
    if cosine is not None and cosine < tau:
        return True, "similarity"
    low = document.lower()
    for m in OVERRIDE_MARKERS:
        if m in low:
            return True, "marker"
    return False, None
''' % (repr(CONFIG["OVERRIDE_MARKERS"]), TAU)

(DIRS["src"] / "module3.py").write_text(MODULE3_SRC)
print("exported src/clinicalshield/module3.py")

module3_results = {
    "notebook": "06_module3_intent",
    "seed": SEED,
    "embed_model": CONFIG["EMBED_MODEL"],
    "protocol": {
        "tau_selected_on": ("validation partition; maximum recall subject to a "
                            "false-positive budget of 5%"),
        "fpr_budget": FPR_BUDGET,
        "tau_selected": float(TAU),
        "tau_v1": CONFIG["TAU_INTENT"],
        "note": ("v1 reported tau = 0.30 without stated justification. The sweep "
                 "runs on validation and the operating point is applied unchanged "
                 "to test. Selection is by recall under an explicit false-positive "
                 "budget rather than by F1: benign and adversarial cosines overlap "
                 "heavily, so F1 is maximised at a threshold that flags most "
                 "documents, which is not a deployable operating point for a CDSS."),
        "query_construction": ("derived from host document metadata (drug and label "
                               "section) only; never from injected payload text"),
        "n_override_markers": len(CONFIG["OVERRIDE_MARKERS"]),
    },
    "similarity_distribution": {
        "benign_mean": float(ben.mean()), "benign_sd": float(ben.std()),
        "adversarial_mean": float(adv.mean()), "adversarial_sd": float(adv.std()),
        "auc_similarity_alone": float(auc_sim),
    },
    "validation_sweep": {
        "similarity_only": {str(k): {kk: vv for kk, vv in v.items() if kk != "reasons"}
                            for k, v in sim_only.items()},
        "combined": {str(k): {kk: vv for kk, vv in v.items() if kk != "reasons"}
                     for k, v in combined.items()},
    },
    "test": {
        "tau": float(TAU),
        "similarity_only": test_sim_only,
        "markers_only": test_marker_only,
        "combined": test_combined,
    },
    "generated_at": time.strftime("%Y-%m-%d %H:%M:%S UTC", time.gmtime()),
}
with open(DIRS["results"] / "module3_results.json", "w") as f:
    json.dump(module3_results, f, indent=2)

if IN_COLAB and DRIVE_ROOT:
    dst = DRIVE_ROOT / "data" / "results"; dst.mkdir(parents=True, exist_ok=True)
    for f_ in (ROOT / "data" / "results").glob("*"):
        shutil.copy2(f_, dst / f_.name)
    print("mirrored to Drive")

if IN_COLAB and CONFIG["PUSH_TO_GITHUB"] and REPO_DIR and REPO_DIR.exists():
    keep = ["data/results/module3_results.json", "src/clinicalshield/module3.py"]
    subprocess.run(["git","-C",str(ROOT),"add","-f"] + keep, check=False)
    st = subprocess.run(["git","-C",str(ROOT),"status","--porcelain"],
                        capture_output=True, text=True)
    if st.stdout.strip():
        subprocess.run(["git","-C",str(ROOT),"commit","-q","-m",
                        "NB06: Module 3, tau=%.2f selected on validation, "
                        "test recall %.4f" % (TAU, test_combined["recall"])], check=False)
        pr = subprocess.run(["git","-C",str(ROOT),"push","-q","origin",
                             CONFIG["GITHUB_BRANCH"]], capture_output=True, text=True)
        print("push:", "ok" if pr.returncode == 0 else pr.stderr[:200])

print("written: module3_results.json")


## Summary

**Paste the output below into the chat.**

The two numbers I will look at first: the AUC on similarity alone, and whether markers-only
differs from both-combined. Between them they determine whether Module 3 is one mechanism or
two.

In [ ]:
# ── Cell 8 · NB06 SUMMARY — paste this output ─────────────────────────────────
print("=" * 68)
print("NB06 — MODULE 3: SEMANTIC INTENT VERIFICATION")
print("=" * 68)
print("embed model : %s" % CONFIG["EMBED_MODEL"].split("/")[-1])
print("tau         : %.2f  (validation, max recall at FPR <= %.0f%%; v1 used %.2f)"
      % (TAU, 100*FPR_BUDGET, CONFIG["TAU_INTENT"]))
print("markers     : %d override phrases" % len(CONFIG["OVERRIDE_MARKERS"]))
print("queries     : built from host metadata only, never from payload")
print("")
print("COSINE SIMILARITY, query vs document")
print("  benign      %.4f +/- %.4f" % (ben.mean(), ben.std()))
print("  adversarial %.4f +/- %.4f" % (adv.mean(), adv.std()))
print("  separation  %.4f" % abs(ben.mean() - adv.mean()))
print("  AUC on similarity alone : %.4f   (0.50 = no signal)" % auc_sim)
print("")
print("HELD-OUT TEST at tau = %.2f" % TAU)
print("configuration          recall     FPR       precision   F1")
for name, s in [("similarity only", test_sim_only),
                ("markers only", test_marker_only),
                ("both (Module 3)", test_combined)]:
    print("  %-20s %.4f   %.4f    %.4f     %.4f"
          % (name, s["recall"], s["fpr"], s["precision"], s["f1"]))
print("")
print("detections by signal : %s" % test_combined["reasons"])
print("")
print("RECALL BY VECTOR (Module 3 alone)")
for vec in ["override", "misinformation"]:
    rows = [d for d in parts["test"] if d["label"] == 1 and d["vector"] == vec]
    if rows:
        hit = sum(flag(d, TAU)[0] for d in rows)
        print("  %-16s n=%4d  %.4f" % (vec, len(rows), hit/len(rows)))
print("")
print("RECALL BY CATEGORY (Module 3 alone)")
for cat in sorted({d["category"] for d in parts["test"] if d["label"] == 1}):
    rows = [d for d in parts["test"] if d["label"] == 1 and d["category"] == cat]
    hit = sum(flag(d, TAU)[0] for d in rows)
    print("  %-28s n=%4d  %.4f" % (cat, len(rows), hit/len(rows)))
print("=" * 68)
print("NB06 COMPLETE — ready for NB07 (Module 4 + CEPS v2)")
print("=" * 68)
